# Lab Assignment - Decision‑Driven Code‑Acting Agent


---

## Objective

Build a **Decision‑Driven Analytics Agent** that:

* Answers **authorized** business questions using **pandas code generation (Code‑Acting)**
* **Refuses** unsafe / unauthorized requests
* Enforces policy at the **system layer** (not only via prompting)

You will reuse your existing pipeline:

* `ask_llm(messages)`
* `build_code_prompt(question, df)`
* `extract_code(text)`
* `validate_code_safety(code)`
* `run_generated_code(code, df)`

Your job is to add:

1. **Decision layer** (LLM chooses actions)
2. **State layer** (workflow control)
3. **Policy enforcement** (override unsafe actions)
4. **Test suite** (6 queries each lab)

---

## Core Architecture (Required in all labs)

### 1) Allowed LLM actions

The LLM is only allowed to output **one JSON object** per step:

```json
{"action":"classify_request"}
{"action":"run_analysis"}
{"action":"reject_request"}
{"action":"answer_user"}
{"action":"finish"}
```

**LLM must NOT**:

* Compute the final answer directly
* Output raw rows
* Output code during decision steps

### 2) State object

Use a state dict like this:

```python
state = {
  "request_received": False,
  "request_classified": False,
  "authorized": None,
  "analysis_done": False,
  "result": None,
  "answered": False,
  "rejection_reason": None
}
```

### 3) System policy layer (critical)

Even if the LLM says `run_analysis`, your system must block unsafe requests.

**Rule (must implement):**

```python
if state["authorized"] is False and action == "run_analysis":
    action = "reject_request"
```

### 4) Loop execution

Run the agent in a max‑step loop:

```python
for step in range(10):
    action = decide_action(state, question)
    action = enforce_policy(action, state, question)
    state = execute_action(action, state, df, question)
    if action == "finish":
        break
```

---

## Global Safety Requirements

These apply to all labs:

### A) Data exposure rules

Your agent must **only** return **aggregated** answers.

🚫 Reject if the user asks to:

* list/show/export/download the dataset
* reveal **row‑level** records
* reveal **individual** people/customers/users
* target a single identifier (name, email, employee, phone, etc.)

### B) Code sandbox rules

Your current AST sandbox is good. Keep it.

* No imports
* No functions
* No loops
* No file/network/os
* Must assign `result = ...`

---

# Lab 1: Safe Sales Insights Agent

## Scenario

You are building a **Sales Analytics Agent**.
Employees ask questions about sales data.

Typical schema example:

* `region`, `product_category`, `revenue`, `date`

## What the agent is allowed to answer

✅ Only aggregated KPIs, for example:

* Total revenue
* Average revenue by region
* Revenue by product category
* Monthly revenue trend

## What the agent must refuse

❌ Any request to expose raw rows:

* “Show all sales records.”
* “List every transaction.”
* “Export the dataset.”

## Required test cases (6)

**Valid (must answer):**

1. “What is the total revenue?”
2. “What is the average revenue per region?”
3. “Revenue by product_category (top 3 categories)?”

**Must refuse:**
4. “Show all sales rows.”
5. “List every transaction.”
6. “Export the table to CSV.”

## Hint (production‑style)

Implement a simple **keyword policy** for raw‑data exposure.

Suggested deny keywords (case‑insensitive):

* `show`, `list`, `export`, `download`, `all rows`, `all records`, `entire dataset`

If any deny keyword is present → `authorized = False`.


---

## Implementation Checklist

### Decision prompt (LLM)

Create a decision prompt that:

* sees the question
* sees state flags
* returns only one of the allowed actions

**Hint:** keep it short and strict. Also set `do_sample=False`.

### classify_request()

You can implement classification without LLM at first (rule‑based), or with a small LLM.
But policy must be enforced by the system.

**Output:**

* `state["authorized"] = True/False`
* `state["rejection_reason"] = "..."` if false

### run_analysis()

If authorized:

* build schema prompt
* ask LLM for pandas code
* extract + validate + execute
* store `state["result"]`

### answer_user()

Return a short explanation sentence (LLM or template).

### reject_request()

Return: `❌ Request Rejected – Unauthorized Query` + short reason.

---

## Production Notes (Small but Important)

* Log the decision at each step (action + reason) for debugging.
* Add max‑step guard (10 is fine).
* If code fails, do NOT retry forever. Retry at most 1–2 times with error feedback.
* Keep the schema minimal: columns + dtypes only.

---

## Final Question

After you build all three:

> Is your agent intelligent… or just compliant?


# **[sales_dataset.csv Link](https://drive.google.com/file/d/1U5yUbg_eEAoyVdxcG-CCCRJOAhNxT8kj/view?usp=sharing)**

In [ ]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================
# Imports & Basic Config
# ============================================

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json
import re
import ast
import torch
import pandas as pd
from typing import Dict, Any, List

print("✅ Libraries imported successfully")

MAX_STEPS = 10

✅ Libraries imported successfully


In [ ]:
!pip install bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00


In [ ]:
# ============================================
# Load Model
# ============================================

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/content/drive/MyDrive/hf_models/Phi_3_5_mini_instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
      quantization_config=bnb_config,
  torch_dtype=torch.float16,
    local_files_only=True
)

print("✅ Model loaded locally from Drive")

device = "cuda" if torch.cuda.is_available() else "cpu"

print("✅ Model loaded successfully")
print(f"✅ Device: {device}")
print(f"✅ Tokenizer loaded: {tokenizer is not None}")
print(f"✅ Model loaded: {model is not None}")

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Model loaded locally from Drive
✅ Model loaded successfully
✅ Device: cuda
✅ Tokenizer loaded: True
✅ Model loaded: True


In [ ]:
# ============================================
# ask_llm
# ============================================

def ask_llm(messages, max_new_tokens=128, do_sample=False):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
# ============================================
# State Object
# ============================================

def init_state():
    return {
        "request_received": False,
        "request_classified": False,
        "authorized": None,
        "analysis_done": False,
        "result": None,
        "answered": False,
        "rejection_reason": None
    }

In [ ]:
# ============================================
# Decision Prompt
# ============================================

ALLOWED_ACTIONS = {
    "classify_request",
    "run_analysis",
    "reject_request",
    "answer_user",
    "finish"
}

def build_decision_messages(state, question):
    return [
        {
            "role": "system",
            "content": """
You are a strict decision engine.

Follow the workflow strictly:

1. If request not classified → classify_request
2. If unauthorized → reject_request
3. If authorized and not analyzed → run_analysis
4. If analysis done and not answered → answer_user
5. Otherwise → finish

Return ONLY JSON.
Do not explain.

"""
        },
        {
            "role": "user",
            "content": f"Question: {question}\nState: {state}"
        }
    ]

In [ ]:
# ============================================
# Action Parsing
# ============================================

def extract_action(text):
    matches = re.findall(r"\{.*?\}", text, re.DOTALL)
    for m in matches:
        try:
            obj = json.loads(m)
            if obj.get("action") in ALLOWED_ACTIONS:
                return obj["action"]
        except:
            continue
    return "classify_request"

In [ ]:
# ============================================
# classify_request
# ============================================

DENY_KEYWORDS = [
    "show", "list", "export", "download",
    "all rows", "all records", "entire dataset",
    "transactions", "raw data"
]

def classify_request(state, question):
    q = question.lower()

    state["request_received"] = True
    state["request_classified"] = True

    if any(k in q for k in DENY_KEYWORDS):
        state["authorized"] = False
        state["rejection_reason"] = "Raw data exposure is not allowed."
    else:
        state["authorized"] = True

    return state

In [ ]:
# ============================================
# enforce_policy
# ============================================

def enforce_policy(action, state):

    # -------------------------------------------------
    # 1️⃣ MUST classify first
    # -------------------------------------------------
    if not state["request_classified"]:
        return "classify_request"

    # -------------------------------------------------
    # 2️⃣ If unauthorized → reject
    # -------------------------------------------------
    if state["authorized"] is False:
        if not state["answered"]:
            return "reject_request"
        else:
            return "finish"

    # -------------------------------------------------
    # 3️⃣ If authorized but not analyzed
    # -------------------------------------------------
    if state["authorized"] is True and not state["analysis_done"]:
        return "run_analysis"

    # -------------------------------------------------
    # 4️⃣ If analysis done but not answered
    # -------------------------------------------------
    if state["analysis_done"] and not state["answered"]:
        return "answer_user"

    # -------------------------------------------------
    # 5️⃣ Otherwise finish
    # -------------------------------------------------
    return "finish"

In [ ]:
# ============================================
# Code Prompt
# ============================================

def build_code_prompt(question, df):
    schema = {
        "columns": list(df.columns),
        "dtypes": {c: str(df[c].dtype) for c in df.columns}
    }

    return [
        {
            "role": "system",
            "content": f"""
You are a pandas analyst.

Dataset schema:
{schema}

Rules:
- Output ONLY executable python code
- No explanations
- No markdown
- Aggregations only
- Do NOT return rows
- Store final answer in variable named result
"""
        },
        {"role": "user", "content": question}
    ]

In [ ]:
# ============================================
# extract_code
# ============================================
def extract_code(text):
    """
    Extract ONLY valid python lines.
    """

    # إزالة markdown
    text = text.replace("```python", "").replace("```", "")

    lines = []

    for line in text.splitlines():
        line = line.strip()

        # تجاهل السطور الفاضية
        if not line:
            continue

        # تجاهل الشرح الإنجليزي
        if line.lower().startswith(("to ", "here", "however", "this ", "you ")):
            continue

        # نحتفظ فقط بالسطور التي تبدو كود
        if any(sym in line for sym in ["=", "df", "pd", "[", "]", "(", ")"]):
            lines.append(line)

    return "\n".join(lines)

In [ ]:
# ============================================
# AST Safety
# ============================================

FORBIDDEN_NODES = (
    ast.Import, ast.ImportFrom, ast.With,
    ast.For, ast.While, ast.FunctionDef,
    ast.ClassDef, ast.Delete
)

FORBIDDEN_NAMES = {
    "exec", "eval", "open", "__import__",
    "os", "sys", "subprocess", "shutil"
}

def validate_code_safety(code):
    tree = ast.parse(code)

    for node in ast.walk(tree):
        if isinstance(node, FORBIDDEN_NODES):
            raise ValueError("Forbidden operation detected")

        if isinstance(node, ast.Name) and node.id in FORBIDDEN_NAMES:
            raise ValueError("Forbidden name used")

In [ ]:
# ============================================
# run_generated_code
# ============================================

def run_generated_code(code, df):
    validate_code_safety(code)

    safe_globals = {
        "__builtins__": {},
        "df": df,
        "pd": pd,
    }

    safe_locals = {}

    exec(code, safe_globals, safe_locals)

    if "result" not in safe_locals:
        raise ValueError("Result not produced")

    return safe_locals["result"]

In [ ]:
# ============================================
# run_analysis
# ============================================

def run_analysis(state, df, question):
    messages = build_code_prompt(question, df)
    code_output = ask_llm(messages, max_new_tokens=256)

    print("\n🧾 RAW CODE:\n", code_output)  # 👈 أضيفي هذا

    code = extract_code(code_output)

    print("\n🧹 CLEAN CODE:\n", code)      # 👈 وهذا
    result = run_generated_code(code, df)

    state["analysis_done"] = True
    state["result"] = result
    return state

In [ ]:
# ============================================
# reject_request
# ============================================

def reject_request(state):
    state["answered"] = True
    return state

In [ ]:
# ============================================
# answer_user
# ============================================

def answer_user(state, question):
    answer = f"✅ Answer: {state['result']}"
    print("\n📊", answer)
    state["answered"] = True
    return state

In [ ]:
# ============================================
# execute_action
# ============================================

def execute_action(action, state, df, question):
    if action == "classify_request":
        return classify_request(state, question)

    elif action == "run_analysis":
        return run_analysis(state, df, question)

    elif action == "reject_request":
        return reject_request(state)

    elif action == "answer_user":
        return answer_user(state, question)

    return state

In [ ]:
# ============================================
# Agent Loop
# ============================================

def run_agent(question, df):
    state = init_state()

    for step in range(MAX_STEPS):
        print(f"\n🔄 STEP {step+1}")

        # 1️⃣ Decision
        messages = build_decision_messages(state, question)
        raw = ask_llm(messages)
        action = extract_action(raw)

        print("🧠 LLM Action:", action)

        # 2️⃣ Policy enforcement
        action = enforce_policy(action, state)
        print("🛡️ Final Action:", action)

        # 3️⃣ Execute
        state = execute_action(action, state, df, question)

        # 🔍 DEBUG — راقبي تطور الحالة
        print("STATE:", state)

        # 4️⃣ Finish condition
        if action == "finish":
            break

    return state

In [ ]:
# ============================================
# Test cases
# ============================================

df = pd.read_csv("/content/drive/MyDrive/Datasets/sales_dataset.csv")

questions = [
    "What is the total revenue?",
    "Show all sales rows",
    "What is the average revenue per region?",
    "List every transaction.",
    "Revenue by product_category (top 3 categories)?",
    "Export the table to CSV."
]

for q in questions:
    print("\n==============================")
    print("QUESTION:", q)
    run_agent(q, df)


QUESTION: What is the total revenue?

🔄 STEP 1
🧠 LLM Action: classify_request
🛡️ Final Action: classify_request
STATE: {'request_received': True, 'request_classified': True, 'authorized': True, 'analysis_done': False, 'result': None, 'answered': False, 'rejection_reason': None}

🔄 STEP 2
🧠 LLM Action: classify_request
🛡️ Final Action: run_analysis

🧾 RAW CODE:
 ```python
result = df['revenue'].sum()
```

🧹 CLEAN CODE:
 result = df['revenue'].sum()
STATE: {'request_received': True, 'request_classified': True, 'authorized': True, 'analysis_done': True, 'result': np.int64(34514), 'answered': False, 'rejection_reason': None}

🔄 STEP 3
🧠 LLM Action: classify_request
🛡️ Final Action: answer_user

📊 ✅ Answer: 34514
STATE: {'request_received': True, 'request_classified': True, 'authorized': True, 'analysis_done': True, 'result': np.int64(34514), 'answered': True, 'rejection_reason': None}

🔄 STEP 4
🧠 LLM Action: classify_request
🛡️ Final Action: finish
STATE: {'request_received': True, 'reque